# Embeddings in a Transformer

Notebook explores how token IDs are mapped into **learned vector repesentations** before they enter the Transformer block.

This includes:
- token embedding table
- tensor shapes
- embedding lookup behavior
- parameter count
- architectural considerations for the actual LLM

In [ ]:
import torch
import torch.nn as nn

## Important Model Decision

### Model Dimension Size
We won't finalize the dimension size, instead making the hyperparameters configurable for the future. \
Some reasonable $d_{model}$ candidate sizes for the actual model however are **256**, **384**, and **512**.

Our vocabulary size grows linearly with the dimension size: $V \times d_{model}$ \
Most Transformer parameters grow like $d^2_{model}$ because of matrix operations.

---

### Attention
The attention block typically grows $4d^2_{model}$ because of:
- query $W_Q$
- key $W_K$
- value $W_V$
- output projection $W_O$

That's where the 4 comes from.

And to project a vector representation of a token $\vec x \rightarrow \vec q,\vec k,\vec v$, you need the projection $W_Q, W_K, W_V$ to be squared. \
The output projection also requires the same $W_O$ matrix to be squared. \
In other words, projecting a length-$d$ vector into another length-$d$ vector, you use a $d×d$ matrix.

That's where the $d^2_{model}$ comes from.

---

### MLP
The MLP/feed-forward network often scales even more agressively with dimension size. \
**up-projection** expands the vector representations into a higher dimension $d_{model} \rightarrow d_{ff}$.

$d_{ff}$ is typically at last 4x the size of $d_{model}$, naturally projection matrix $W_\uparrow$ needs to match that size, squared.

## Code and Notes

In order for the embedding table and everything else can be inspectable, we'll use tiny dimensions in `nn.Embedding()`

https://docs.pytorch.org/docs/2.14/generated/torch.nn.Embedding.html

In [ ]:
vocab_size = 16
d_model = 8

embedding = nn.Embedding(
    num_embeddings=vocab_size, # number of embeddings <-> vocab size
    embedding_dim=d_model # embedding dimension
)

print(embedding)

We get the shape of the embedding tensor and total number of elements (parameters) in the tensor with `numel()`

https://docs.pytorch.org/docs/2.14/generated/torch.numel.html

In [ ]:
print("Embedding weight shape:", embedding.weight.shape)
print("Parameter count:", embedding.weight.numel()) # total elements in tensor

In [ ]:
# Sample batch of token IDs
token_ids = torch.tensor([
    [1, 2, 3, 4],
    [5, 6, 7, 1]
])

print("Token IDs:")
print(token_ids)

print("\nShape:", token_ids.shape)

We get our first instance of $ [B, T, d_{model}] $

In [ ]:
# convert IDs to embedding vectors
x = embedding(token_ids)

print("Embedding output shape:", x.shape)

print("\nTensor:\n", x)

In [ ]:
# First token in first batch
token_id = token_ids[0, 0]

# Get embedding in embedding matrix
manual_lookup = embedding.weight[token_id]

# Get embedding from model
model_lookup = x[0, 0]

print("Token ID:", token_id)
print("\nManual lookup:\n", manual_lookup)

print("\nEmbedding output:\n", model_lookup)

print("\nSame Embeddings:", torch.equal(manual_lookup, model_lookup))

We can see below that embedding vectors alone do **not** inherently have positional information nor surrounding contextual information.

In [ ]:
# token 1 in first batch
b1_one = x[0, 0]

# token 1 in second batch
b2_one = x[1, 3]

print("First batch's token 1 embedding:\n", b1_one)

print("\nSecond batch's token 1 embedding:\n", b2_one)

print("\nExact same embeddings:", torch.equal(b1_one, b2_one))

### Embedding Gradients

We'll make a **fake** sample loss score by simply summing up the embeddings, then perform backprop to get the gradients.

https://docs.pytorch.org/docs/2.14/generated/torch.optim.Optimizer.zero_grad.html \
https://docs.pytorch.org/docs/2.14/generated/torch.Tensor.backward.html


In [ ]:
# Clear gradients so they don't accumulate
embedding.zero_grad()

x = embedding(token_ids)

# Fake scalar loss only for demonstrating gradient flow
loss = x.sum()

print("Loss:", loss)

# Backprop (compute gradients)
loss.backward()

print("\nGradient shape:", embedding.weight.grad.shape)

Notice the resulting gradients have shape $[V,d_{model}]$, only rows selected in `token_ids` receive non-zero gradients. Token `1` has a higher gradient as they contribute to the computation twice in the batch.

In [ ]:
print("Gradient for token 1:")
print(embedding.weight.grad[1])

print("\nGradient for token 7:")
print(embedding.weight.grad[7])

print("\nGradient for token 8, which is not in batch:")
print(embedding.weight.grad[8])

### Embedding Class

All of this can be wrapped in a class that has the previous code logic automatically run.

In [ ]:
# Class that maps token IDs to d_model-dimensional vectors
class TokenEmbedding(nn.Module):
    """
    Maps token IDs to d_model size vectors
    """

    # Constructor
    def __init__(self, vocab_size, d_model):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=d_model
        )

    # Embed token IDs 
    def forward(self, token_ids):
        return self.embedding(token_ids)

In [ ]:
token_embedding = TokenEmbedding(
    vocab_size=vocab_size,
    d_model=d_model
)

x = token_embedding(token_ids)

print("Input shape:", token_ids.shape)
print("Output shape:", x.shape)

## Summary

So far, we created an `nn.Embedding` table with shape $[V,d_{model}]$, where each row is the learned vector representation for one token.

`token_ids` shaped $[B,T]$ are passed through and embedded, resulting in embedding vector representations, shaped $[T,B,d_{model}]$.

Those embeddings were verified that the embedding lookup was simply just selecting a row from `embedding.weight` that matches a token ID.

The embedding table was contained learnable parameters, as tokens used in the batch received gradients via backprop `backward()` and repeating ones accumulated gradient contributions to the same row.

## Important Model Decision

#### Context
In the embedding stage, the embedding lookup matrix is $E \in \mathbb{R}^{V \times d_{model}}$

Near the final stages of the forward-pass, the LM head maps the final output $[B,T,d_{model}] \rightarrow [B,T,V]$

Conceptually, that output projection matrix must be $\in \mathbb{R}^{V \times d_{model}}$, in other words, shaped $[d_{model},V]$

---

#### Weight Tying
**Tied**: reuse the embedding matrix for LM head, using its transpose $E^T$ \
**Not tied**: input embedding weights and LM head weights are **separate** parameters.

Weight tying reuses the same learned embedding table for both input and output:
- **Input:** look up each token ID’s embedding vector.
- **Output:** multiply the final hidden vectors by the transposed embedding table to produce a logit for each vocabulary token.

This also reduces parameters, suppose the embedding matrix has $32,000 \times 512 = 16,384,000$ parameters. Tying would avoid duplicating parameters, saving **~16.4 million** parameters.

**Without** weight tying:
- embedding matrix: $V \times d_{model}$
- LM head projection matrix: $d_{model} \times V$

**With** weight tying:
- embedding matrix: $E \in \mathbb{R}^{V \times d_{model}}$
- LM head: $E^T$

---

#### For a relatively small model, this could save a significant amount of parameters.

In [ ]:
vocab_size = 16
d_model = 8

embedding = nn.Embedding(vocab_size, d_model) # embedding table shape [V,d_model]

hidden = torch.randn(2, 4, d_model) # [B,T,d_model]

# Matrix multiplication for projection
logits = hidden @ embedding.weight.T 

print("Hidden shape:", hidden.shape)
print("Embedding weight shape:", embedding.weight.shape)
print("Logits shape:", logits.shape)

## Final Verdict

#### USE WEIGHT TYING AS THE BENEFITS FAR OUTWEIGH THE FLEXBILITY IT PROVIDES

Weight tying should be used for the model that I am trying to build, as the **embedding/LM head** parameters will most likely take up a huge part of the entire model's total parameters.

---

The main benefit of **not** using weight tying is that the model's input embedding matrix and output projection matrix are allowed different learned representations. This **might** result in marginally better performance in large models that have the paramter capacity to not require weight tying, as the input embeddings and output projection can specialize instead of relying on a single learned matrix to perform both jobs.

However, the performance gain does **not** outweigh the embedding/LM head parameters effectively doubling, especially for a small model that I am trying to build. Most of the time, the difference in performance is negligible and not meaningfully improve performance. The biggest benefit is flexibility because of the 2 distinct matrices.

---

Ultimately, whether that freedom of **not** tying effectively translates into a better language model is an **empirical architecture choice**.
